1. Imports

In [2]:
import os
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.cluster import AgglomerativeClustering as AggCls
from datetime import datetime
import numpy as np
from collections import Counter
import pandas as pd
import csv
import ast
from sklearn.cluster import KMeans
from transformers import BertTokenizer, BertModel
import torch
import torch.nn.functional as F

c:\Users\korey\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [ ]:
clozeDf = pd.read_csv("../cloze_winter_2018_val.csv", sep = ",")

clozeDf.head(5)

,InputStoryid,InputSentence1,InputSentence2,InputSentence3,InputSentence4,RandomFifthSentenceQuiz1,RandomFifthSentenceQuiz2,AnswerRightEnding
0,138d5bfb-05cc-41e3-bf2c-fa85ebad14e2,Rick grew up in a troubled household.,"He never found good support in family, and tur...",It wasn't long before Rick got shot in a robbery.,The incident caused him to turn a new leaf.,He is happy now.,He joined a gang.,1
1,bff9f820-9605-4875-b9af-fe6f14d04256,Laverne needs to prepare something for her fri...,She decides to bake a batch of brownies.,She chooses a recipe and follows it closely.,Laverne tests one of the brownies to make sure...,The brownies are so delicious Laverne eats two...,Laverne doesn't go to her friend's party.,1
2,e8f628d5-9f97-40ed-8611-fc0e774673c4,Sarah had been dreaming of visiting Europe for...,She had finally saved enough for the trip.,She landed in Spain and traveled east across t...,She didn't like how different everything was.,Sarah then decided to move to Europe.,Sarah decided that she preferred her home over...,2
3,f5226bfe-9f26-4377-b05f-3d9568dbdec1,Gina was worried the cookie dough in the tube ...,She was very happy to find she was wrong.,The cookies from the tube were as good as from...,Gina intended to only eat 2 cookies and save t...,Gina liked the cookies so much she ate them al...,Gina gave the cookies away at her church.,1
4,69ac9b05-b956-402f-9fff-1f926ef9176b,It was my final performance in marching band.,I was playing the snare drum in the band.,We played Thriller and Radar Love.,The performance was flawless.,I was very proud of my performance.,I was very ashamed of my performance.,1


In [3]:
from sentence_transformers import SentenceTransformer, util

In [4]:
model = SentenceTransformer("all-MiniLM-L6-v2") #S-BERT Model

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 4896.33it/s]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


2. Clustering Formulation Example

- K-Means Clustering with S-BERT

Narrative labelled '0e499832-cc4e-459d-a066-de6e73aa77f9' from ROCStories corpus (Mostafazadeh et al., 2016). The narrative has a labelled 'Correct' ending as 2, and utilizing kmeans interia, the narrative passage containing the labelled 'Correct' ending has kmeans interia of 2.70, whereas with the labelled 'Inorrect' ending the kmeans interia is 2.73. Therefore, showing a case where smaller kmeans interia (sum of squared cluster distance) may be assigned to clusters including the labelled 'correct' sentence compared to a cluster keeping the same four narrative sentences, but replacing the ending with an 'incorrect' labelled sentence.

    - Ending 1


In [9]:
sentences = ["Jay was finishing up lunch at McDonald's, throwing away his trash.", "As soon as he threw it away, he realized there was a promo happening.", "There were some game pieces on his fry box so he fished it out.", "He peeled off the game piece and couldn't believe it.", "Jay thought Burger King was awesome."]
embeddings = model.encode(sentences)

kmeans = KMeans(n_clusters=1, init='k-means++', random_state=42)
clusterLabels = kmeans.fit_predict(embeddings)
#kmeans.fit(embeddings)
print(clusterLabels)
print(kmeans.inertia_)

[0 0 0 0 0]
2.7360997200012207


    - Ending 2

In [6]:
sentences = ["Jay was finishing up lunch at McDonald's, throwing away his trash.", "As soon as he threw it away, he realized there was a promo happening.", "There were some game pieces on his fry box so he fished it out.", "He peeled off the game piece and couldn't believe it.", "He'd dug in the trash for nothing."]
embeddings = model.encode(sentences)

# Assume 'embeddings' is your (n_samples, 756) array
kmeans = KMeans(n_clusters=1, init='k-means++', random_state=42)
clusterLabels = kmeans.fit_predict(embeddings)

print(clusterLabels) #All sentences in one cluster.
print(kmeans.inertia_) #Sum of distances squared from centroid.

[0 0 0 0 0]
2.703484535217285


3. K-Means Clustering with LLMs

3.1. S-BERT K-Means Clustering

In [ ]:
with open('./rocStoriesSpreadsheets/sbertKmeansClusteringDistances.csv', 'w', newline = '') as file: #This cell writes the output embedding vectors produced from S-BERT to a dataset for future processing.
    writer = csv.writer(file)

    writer.writerow(['Story ID', 'Sentences', 'First Ending Sent','First Ending Kmeans Distance Metric', 'Second Ending Sent', 'Second Ending KMeans Distance Metric','Correct Ending'])

    sentences = []
    for i, row in clozeDf.iterrows():

        ##ENDING ONE

        #Assembling Sentences & Formulating Embeddings
        sentenceHolder = row['InputSentence1']
        sentences.append(str(sentenceHolder))

        sentenceHolder = row['InputSentence2']
        sentences.append(str(sentenceHolder))

        sentenceHolder = row['InputSentence3']
        sentences.append(str(sentenceHolder))

        sentenceHolder = row['InputSentence4']
        sentences.append(str(sentenceHolder))

        sentenceHolder = row['RandomFifthSentenceQuiz1']
        sentences.append(str(sentenceHolder))

        
        embeddings = model.encode(sentences, normalize_embeddings=True)
        
        #Kmeans Clustering
        
        kmeans = KMeans(n_clusters=1, init='k-means++', n_init = 'auto', random_state=42)

        clusterLabels = kmeans.fit_predict(embeddings)
        
        #Getting distance metric.

        #print(sentences)
        #print(cluster_labels)
        #print(kmeans.inertia_)

        firstDistanceMetric = kmeans.inertia_
        sentences = []

        ##ENDING TWO

        sentenceHolder = row['InputSentence1']
        sentences.append(str(sentenceHolder))

        sentenceHolder = row['InputSentence2']
        sentences.append(str(sentenceHolder))

        sentenceHolder = row['InputSentence3']
        sentences.append(str(sentenceHolder))

        sentenceHolder = row['InputSentence4']
        sentences.append(str(sentenceHolder))

        sentenceHolder = row['RandomFifthSentenceQuiz2']
        sentences.append(str(sentenceHolder))

        
        embeddings = model.encode(sentences, normalize_embeddings=True)
        
        #Kmeans Clustering
        
        kmeans = KMeans(n_clusters=1, init='k-means++', n_init = 'auto', random_state=42)

        clusterLabels = kmeans.fit_predict(embeddings)
        
        #Getting distance metric.

        #print(sentences)
        #print(cluster_labels)
        #print(kmeans.inertia_)
        secondDistanceMetric = kmeans.inertia_
        writer.writerow([row['InputStoryid'], sentences[:-1], row['RandomFifthSentenceQuiz1'],firstDistanceMetric, row['RandomFifthSentenceQuiz2'], secondDistanceMetric, row['AnswerRightEnding']])

        sentences = []
        if i % 100 == 0:
            print('Number of Stories Processed: {}'.format(i))
        
        



Number of Stories Processed: 0
Number of Stories Processed: 100
Number of Stories Processed: 200
Number of Stories Processed: 300
Number of Stories Processed: 400
Number of Stories Processed: 500
Number of Stories Processed: 600
Number of Stories Processed: 700
Number of Stories Processed: 800
Number of Stories Processed: 900
Number of Stories Processed: 1000
Number of Stories Processed: 1100
Number of Stories Processed: 1200
Number of Stories Processed: 1300
Number of Stories Processed: 1400
Number of Stories Processed: 1500


In [ ]:
with open('./rocStoriesSpreadsheets/sbertKmeansClusteringDistances.csv', mode='r', newline='', encoding='utf-8') as csvFile: #This cell reads in the existing dataset and calculates the accuracy of narrative ending selection by the distance-based formulation.
    
    csvReader = csv.reader(csvFile, delimiter=',') 

    correctCount = 0
    incorrectCount = 0

    i = 0
    for row in csvReader:
        if i != 0:
            #print(row)
            if int(row[6]) == 1:
                if float(row[3]) < float(row[5]):
                   correctCount += 1
                if float(row[5]) < float(row[3]):
                   incorrectCount += 1

            if int(row[6]) == 2:
                if float(row[3]) < float(row[5]):
                   incorrectCount += 1
                if float(row[5]) < float(row[3]):
                   correctCount += 1
            #print(row[3], row[5])
            #print("Correct Count: {} | Incorrect Count: {}".format(correct_count, incorrect_count))

        
        i = i + 1
        #if i > 1:
        #    break
        

In [4]:
print("S-BERT Model Clustering, Correctly Identifies Matching Narrative Ending\n - Correct Narrative Ending Selected (n = {} times)\n - Incorrect Narrative Ending Selected (n = {} times).\nS-BERT Model Accuracy: {}".format(correctCount, incorrectCount, correctCount / (incorrectCount + correctCount)))

S-BERT Model Clustering, Correctly Identifies Matching Narrative Ending
 - Correct Narrative Ending Selected (n = 964 times)
 - Incorrect Narrative Ending Selected (n = 607 times).
S-BERT Model Accuracy: 0.6136218968809676


3.2. S3BERT K-Means Clustering Analysis

In [ ]:
s3bertDf = pd.read_excel("../s3bert_k_means_vectors.xlsx")



In [13]:
s3bertDf.head(5)

,FirstSent,SecondSent,ThirdSent,FourthSent,FirstEnd,SecondEnd,FirstSentVec,SecondSentVec,ThirdSentVec,FourthSentVec,FirstEndVec,SecondEndVec
0,Rick grew up in a troubled household.,"He never found good support in family, and tur...",It wasn't long before Rick got shot in a robbery.,The incident caused him to turn a new leaf.,He is happy now.,He joined a gang.,[[-7.14868540e-03 6.55308887e-02 3.37731019e...,[[ 6.89292848e-02 1.23088226e-01 -3.84106487e...,[[-1.92712073e-03 6.48251325e-02 -2.06614453e...,[[ 0.06974935 0.05996247 0.00567702 -0.02274...,[[ 4.00534160e-02 6.00098111e-02 1.35483295e...,[[ 5.36929369e-02 6.23688400e-02 -3.87043431e...
1,Laverne needs to prepare something for her fri...,She decides to bake a batch of brownies.,She chooses a recipe and follows it closely.,Laverne tests one of the brownies to make sure...,The brownies are so delicious Laverne eats two...,Laverne doesn't go to her friend's party.,[[-3.09658721e-02 2.90768296e-02 5.35883568e...,[[-4.97298576e-02 -4.55511622e-02 3.82606760e...,[[ 1.30745471e-02 2.04028860e-02 7.20624849e...,[[-2.95109563e-02 -7.36537250e-03 2.55035982e...,[[-2.11811438e-02 -5.44521250e-02 5.71314618e...,[[ 3.33979726e-02 8.77757091e-03 3.27234231e...
2,Sarah had been dreaming of visiting Europe for...,She had finally saved enough for the trip.,She landed in Spain and traveled east across t...,She didn't like how different everything was.,Sarah then decided to move to Europe.,Sarah decided that she preferred her home over...,[[ 9.05410852e-03 -1.24337031e-02 5.75685687e...,[[ 5.97167611e-02 -3.41265574e-02 8.97795781e...,[[ 8.69741198e-03 -5.89529090e-02 1.54365366e...,[[-2.38292515e-02 1.80734731e-02 6.85321242e...,[[ 7.85042252e-03 -3.78496237e-02 1.12723568e...,[[ 0.03311509 -0.02591359 0.00954012 -0.00951...
3,Gina was worried the cookie dough in the tube ...,She was very happy to find she was wrong.,The cookies from the tube were as good as from...,Gina intended to only eat 2 cookies and save t...,Gina liked the cookies so much she ate them al...,Gina gave the cookies away at her church.,[[-3.10635809e-02 4.02618237e-02 6.22956827e...,[[ 2.51520853e-02 -1.52715505e-03 8.67926702e...,[[-4.64290939e-02 -2.15550177e-02 9.89392772e...,[[ 2.27709450e-02 1.87909845e-02 3.36822160e...,[[ 4.34935745e-03 6.29451359e-03 8.08348730e...,[[-4.91574742e-02 5.20036668e-02 5.37184663e...
4,It was my final performance in marching band.,I was playing the snare drum in the band.,We played Thriller and Radar Love.,The performance was flawless.,I was very proud of my performance.,I was very ashamed of my performance.,[[ 2.41455007e-02 8.96569043e-02 8.22737366e...,[[ 3.58655825e-02 -3.38232331e-03 7.62091652e...,[[ 1.54526997e-03 2.24248716e-03 -2.37030424e...,[[ 3.08983643e-02 2.12712102e-02 1.98132433e...,[[ 1.86308064e-02 7.86623880e-02 4.93844189e...,[[ 4.79168035e-02 3.33064049e-02 1.15768099e...


In [ ]:
import pandas as pd
import ast
import re

def convertStringToList(s):
    s = s.strip()
    s = re.sub(r'\s+', ',', s)
    s = s.replace('[,', '[').replace(',]', ']')
    return np.array(ast.literal_eval(s))

s3bertDf['fittedFirstSentVec'] = s3bertDf['FirstSentVec'].apply(convertStringToList)
s3bertDf['fittedSecondSentVec'] = s3bertDf['SecondSentVec'].apply(convertStringToList)
s3bertDf['fittedThirdSentVec'] = s3bertDf['ThirdSentVec'].apply(convertStringToList)
s3bertDf['fittedFourthSentVec'] = s3bertDf['FourthSentVec'].apply(convertStringToList)
s3bertDf['fittedFirstEndVec'] = s3bertDf['FirstEndVec'].apply(convertStringToList)
s3bertDf['fittedSecondEndVec'] = s3bertDf['SecondEndVec'].apply(convertStringToList)

# Verify the conversion
print(s3bertDf['fittedFirstSentVec'].head(5))
print(type(s3bertDf['fittedFirstSentVec'].iloc[0])) # Should now be '<class \'list\'>' (or similar)

0    [[-0.0071486854, 0.0655308887, 0.0337731019, -...
1    [[-0.0309658721, 0.0290768296, 0.0535883568, -...
2    [[0.00905410852, -0.0124337031, 0.0575685687, ...
3    [[-0.0310635809, 0.0402618237, 0.0622956827, 0...
4    [[0.0241455007, 0.0896569043, 0.0822737366, -0...
Name: fittedFirstSentVec, dtype: object
<class 'numpy.ndarray'>


In [ ]:
with open('./rocStoriesSpreadsheets/s3bertKmeansClusteringDistances.csv', 'w', newline = '') as file:
    writer = csv.writer(file)

    writer.writerow(['First Ending Sent','First Ending Kmeans Distance Metric', 'Second Ending Sent', 'Second Ending KMeans Distance Metric'])

    for i, row in s3bertDf.iterrows():
    
        ##FIRST ENDING

        kmeans = KMeans(n_clusters=1, init='k-means++', n_init = 'auto', random_state=42)
        #print(row['fittedFirstSentVec'].shape)

        x = F.normalize(torch.from_numpy(row['fittedFirstSentVec']), p = 2, dim = 1).reshape(1, -1)
        x2 = F.normalize(torch.from_numpy(row['fittedSecondSentVec']), p = 2, dim = 1).reshape(1, -1)
        x3 = F.normalize(torch.from_numpy(row['fittedThirdSentVec']), p = 2, dim = 1).reshape(1, -1)
        x4 = F.normalize(torch.from_numpy(row['fittedFourthSentVec']), p = 2, dim = 1).reshape(1, -1)
        x5 = F.normalize(torch.from_numpy(row['fittedFirstEndVec']), p = 2, dim = 1).reshape(1, -1)
        #print(x[0])

        clusterLabels = kmeans.fit_predict([x[0], x2[0], x3[0], x4[0], x5[0]])
        firstDistanceMetric = kmeans.inertia_

        #print(row['FirstSent'])
        #print(row['FirstEnd'])

        #print("First Ending Cluster: {}".format(cluster_labels))

        #print("First Ending Cluser Distance Metric: {}".format(distance_metric))

        ##SECOND ENDING

        kmeans = KMeans(n_clusters=1, init='k-means++', n_init = 'auto', random_state=42)

        x = F.normalize(torch.from_numpy(row['fittedFirstSentVec']), p = 2, dim = 1).reshape(1, -1)
        x2 = F.normalize(torch.from_numpy(row['fittedSecondSentVec']), p = 2, dim = 1).reshape(1, -1)
        x3 = F.normalize(torch.from_numpy(row['fittedThirdSentVec']), p = 2, dim = 1).reshape(1, -1)
        x4 = F.normalize(torch.from_numpy(row['fittedFourthSentVec']), p = 2, dim = 1).reshape(1, -1)
        x5 = F.normalize(torch.from_numpy(row['fittedSecondEndVec']), p = 2, dim = 1).reshape(1, -1)
        #print(x[0])

        clusterLabels = kmeans.fit_predict([x[0], x2[0], x3[0], x4[0], x5[0]])
        secondDistanceMetric = kmeans.inertia_

        #print(row['FirstSent'])
        #print(row['SecondEnd'])

        #print("Second Ending Cluster: {}".format(cluster_labels))

        #print("Second Ending Cluser Distance Metric: {}".format(distance_metric))

        #print(cluster_labels)
        #print(row['fittedFirstEndVec'][0])

        writer.writerow([row['FirstEnd'], firstDistanceMetric, row['SecondEnd'], secondDistanceMetric])
        if i % 100 == 0:
            print("{} Stories Processed".format(i))
    

0 Stories Processed
100 Stories Processed
200 Stories Processed
300 Stories Processed
400 Stories Processed
500 Stories Processed
600 Stories Processed
700 Stories Processed
800 Stories Processed
900 Stories Processed
1000 Stories Processed
1100 Stories Processed
1200 Stories Processed
1300 Stories Processed
1400 Stories Processed
1500 Stories Processed


In [5]:
sbertDf = pd.read_csv("./rocStoriesSpreadSheets/sbertKmeansClusteringDistances.csv")
sbertDf.head(5)

correctAnsList = list(sbertDf['Correct Ending'])
print(len(correctAnsList))

1571


In [6]:
with open('./rocStoriesSpreadsheets/s3bertKmeansClusteringDistances.csv') as csv_file:
    csv_reader = csv.reader(csv_file, delimiter=',') 

    correct_count = 0
    incorrect_count = 0

    i = 0
    for row in csv_reader:
        if i != 0:
            #print(i)
            if correctAnsList[i-1] == 1: #Note this value is subtracted by 1 since the correct_ans_list is a separate array declared elsewhere. The (i-1) factor accounts for the excel header.
                if float(row[1]) < float(row[3]):
                    correct_count += 1
                if float(row[3]) < float(row[1]):
                    incorrect_count += 1
            if correctAnsList[i-1] == 2:
                if float(row[1]) < float(row[3]):
                    incorrect_count += 1
                if float(row[3]) < float(row[1]):
                    correct_count += 1
        
        i = i + 1

In [7]:
print("S3BERT Model Clustering, Correctly Identifies Matching Narrative Ending\n - Correct Narrative Ending Selected (n = {} times)\n - Incorrect Narrative Ending Selected (n = {} times).\nS3BERT Model Accuracy: {}".format(correct_count, incorrect_count, correct_count / (correct_count + incorrect_count)))

S3BERT Model Clustering, Correctly Identifies Matching Narrative Ending
 - Correct Narrative Ending Selected (n = 1019 times)
 - Incorrect Narrative Ending Selected (n = 552 times).
S3BERT Model Accuracy: 0.648631444939529


3.3. BERT Model K-Means Clustering

In [ ]:
tokenizer = BertTokenizer.from_pretrained('bert-base-uncased')
model = BertModel.from_pretrained('bert-base-uncased')

In [ ]:
with open('bertKmeansClusteringDistances.csv', 'w', newline = '') as file:
    writer = csv.writer(file)

    writer.writerow(['StoryID','First Ending Sent','First Ending Kmeans Distance Metric', 'Second Ending Sent', 'Second Ending KMeans Distance Metric', 'Correct Answer'])

    #Init tokenizer

    for i, row in clozeDf.iterrows():
     

##GET BERT EMBEDDINGS FIRST ENDING
#Narrative sentences

        sentence1 = str(row['InputSentence1'])
        sentence2 = str(row['InputSentence2'])
        sentence3 = str(row['InputSentence3'])
        sentence4 = str(row['InputSentence4'])

        sentence5 = str(row['RandomFifthSentenceQuiz1'])

# Tokenize the sentences
        tokens1 = tokenizer.tokenize(sentence1)
        tokens2 = tokenizer.tokenize(sentence2)
        tokens3 = tokenizer.tokenize(sentence3)
        tokens4 = tokenizer.tokenize(sentence4)
         
        tokens5 = tokenizer.tokenize(sentence5)

        tokens = ['[CLS]'] + tokens1 + ['[SEP]'] + tokens2 + ['[SEP]'] + tokens3 + ['[SEP]'] + tokens4 + ['[SEP]'] + tokens5
      
        inputIds = tokenizer.convert_tokens_to_ids(tokens)

# Convert tokens to input IDs
        inputIds1 = torch.tensor(tokenizer.convert_tokens_to_ids(tokens1)).unsqueeze(0) 
        inputIds2 = torch.tensor(tokenizer.convert_tokens_to_ids(tokens2)).unsqueeze(0)  
        inputIds3 = torch.tensor(tokenizer.convert_tokens_to_ids(tokens3)).unsqueeze(0)  
        inputIds4 = torch.tensor(tokenizer.convert_tokens_to_ids(tokens4)).unsqueeze(0)  

        inputIds5 = torch.tensor(tokenizer.convert_tokens_to_ids(tokens5)).unsqueeze(0)  

# BERT embeddings
        with torch.no_grad():
                outputs1 = model(inputIds1)
                outputs2 = model(inputIds2)
                outputs3 = model(inputIds3)
                outputs4 = model(inputIds4)

                outputs5 = model(inputIds5)

        embeddings1 = F.normalize(outputs1.last_hidden_state[:, 0, :], p = 2, dim = 1)  
        embeddings2 = F.normalize(outputs2.last_hidden_state[:, 0, :], p = 2, dim = 1)  
        embeddings3 = F.normalize(outputs3.last_hidden_state[:, 0, :], p = 2, dim = 1)  
        embeddings4 = F.normalize(outputs4.last_hidden_state[:, 0, :], p = 2, dim = 1)  
        
        embeddings5 = F.normalize(outputs5.last_hidden_state[:, 0, :], p = 2, dim = 1)  

        kmeans = KMeans(n_clusters=1, init='k-means++', n_init = 'auto', random_state=42)

        x = embeddings1[0].reshape(1, -1)
        x2 = embeddings2[0].reshape(1, -1)
        x3 = embeddings3[0].reshape(1, -1)
        x4 = embeddings4[0].reshape(1, -1)
        x5 = embeddings5[0].reshape(1, -1)

#Clustering
    
        clusterLabels = kmeans.fit_predict([x[0], x2[0], x3[0], x4[0], x5[0]])
        distanceMetricOne = kmeans.inertia_
        #print(sentence5, cluster_labels, distance_metric_one)

##SECOND ENDING

#Narrative sentences

        sentence1 = str(row['InputSentence1'])
        sentence2 = str(row['InputSentence2'])
        sentence3 = str(row['InputSentence3'])
        sentence4 = str(row['InputSentence4'])

        sentence5 = str(row['RandomFifthSentenceQuiz2'])

# Tokenize the sentences
        tokens1 = tokenizer.tokenize(sentence1)
        tokens2 = tokenizer.tokenize(sentence2)
        tokens3 = tokenizer.tokenize(sentence3)
        tokens4 = tokenizer.tokenize(sentence4)
         
        tokens5 = tokenizer.tokenize(sentence5)

        tokens = ['[CLS]'] + tokens1 + ['[SEP]'] + tokens2 + ['[SEP]'] + tokens3 + ['[SEP]'] + tokens4 + ['[SEP]'] + tokens4
      
        input_ids = tokenizer.convert_tokens_to_ids(tokens)

# Convert tokens to input IDs
        inputIds1 = torch.tensor(tokenizer.convert_tokens_to_ids(tokens1)).unsqueeze(0)  
        inputIds2 = torch.tensor(tokenizer.convert_tokens_to_ids(tokens2)).unsqueeze(0) 
        inputIds3 = torch.tensor(tokenizer.convert_tokens_to_ids(tokens3)).unsqueeze(0)  
        inputIds4 = torch.tensor(tokenizer.convert_tokens_to_ids(tokens4)).unsqueeze(0) 

        inputIds5 = torch.tensor(tokenizer.convert_tokens_to_ids(tokens5)).unsqueeze(0)  

# BERT embeddings
        with torch.no_grad():
                outputs1 = model(inputIds1)
                outputs2 = model(inputIds2)
                outputs3 = model(inputIds3)
                outputs4 = model(inputIds4)

                outputs5 = model(inputIds5)

        embeddings1 = F.normalize(outputs1.last_hidden_state[:, 0, :], p = 2, dim = 1)  
        embeddings2 = F.normalize(outputs2.last_hidden_state[:, 0, :], p = 2, dim = 1)  
        embeddings3 = F.normalize(outputs3.last_hidden_state[:, 0, :], p = 2, dim = 1)  
        embeddings4 = F.normalize(outputs4.last_hidden_state[:, 0, :], p = 2, dim = 1)  
    
        embeddings5 = F.normalize(outputs5.last_hidden_state[:, 0, :], p = 2, dim = 1)  

#Clustering

        kmeans = KMeans(n_clusters=1, init='k-means++', n_init = 'auto', random_state=42)

        x = embeddings1[0].reshape(1, -1)
        x2 = embeddings2[0].reshape(1, -1)
        x3 = embeddings3[0].reshape(1, -1)
        x4 = embeddings4[0].reshape(1, -1)
        x5 = embeddings5[0].reshape(1, -1)
    
        clusterLabels = kmeans.fit_predict([x[0], x2[0], x3[0], x4[0], x5[0]])
        distanceMetricTwo = kmeans.inertia_
        #print(sentence5, cluster_labels, distance_metric_two)


        writer.writerow([row['InputStoryid'], str(row['RandomFifthSentenceQuiz1']), distanceMetricOne, str(row['RandomFifthSentenceQuiz2']), distanceMetricTwo, row['AnswerRightEnding']])
        
        if i % 100 == 0:
                print("{} Stories Processed".format(i))
    #print(embeddings1)

0 Stories Processed
100 Stories Processed
200 Stories Processed
300 Stories Processed
400 Stories Processed
500 Stories Processed
600 Stories Processed
700 Stories Processed
800 Stories Processed
900 Stories Processed
1000 Stories Processed
1100 Stories Processed
1200 Stories Processed
1300 Stories Processed
1400 Stories Processed
1500 Stories Processed


In [8]:
with open("./rocStoriesSpreadsheets/bertKmeansClusteringDistances.csv", mode='r', newline='', encoding='utf-8') as csvFile:
    
    csvReader = csv.reader(csvFile, delimiter=',') 

    correctCount = 0
    incorrectCount = 0

    i = 0
    for row in csvReader:
        if i != 0:
            #print(row)
            if int(row[5]) == 1:
                if float(row[2]) < float(row[4]):
                   correctCount += 1
                if float(row[4]) < float(row[2]):
                   incorrectCount += 1

            if int(row[5]) == 2:
                if float(row[2]) < float(row[4]):
                   incorrectCount += 1
                if float(row[4]) < float(row[2]):
                   correctCount += 1

        i = i + 1
        

In [9]:
print("BERT Model Clustering, Correctly Identifies Matching Narrative Ending\n - Correct Narrative Ending Selected (n = {} times)\n - Incorrect Narrative Ending Selected (n = {} times).\nBERT Model Accuracy: {}".format(correctCount, incorrectCount, correctCount / (incorrectCount + correctCount)))

BERT Model Clustering, Correctly Identifies Matching Narrative Ending
 - Correct Narrative Ending Selected (n = 762 times)
 - Incorrect Narrative Ending Selected (n = 809 times).
BERT Model Accuracy: 0.48504137492043287


References

Mostafazadeh, N., Chambers, N., He, X., Parikh, D., Batra, D., Vanderwende, L., Kohli, P., & Allen, J. (2016). “A Corpus and Cloze Evaluation for Deeper Understanding of Commonsense Stories.” Proceedings of the 2016 Conference of the North American Chapter of the Association for Computational Linguistics: Human Language Technologies. Association for Computational Linguistics. pp. 839 – 849. https://aclanthology.org/N16-1098/
<br>
Scikit-Learn Developers. (2026). “Kmeans” [Documentation]. https://scikit-learn.org/stable/modules/generated/sklearn.cluster.KMeans.html

